# Capgemini India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** capgemini.com/in-en/careers

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-03-31 21:41:00
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "Capgemini"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Capgemini/Outputs/2026_03_31


In [4]:
print("=" * 60)
print("CAPGEMINI INDIA JOB SCRAPER")
print("Primary: Workday API + Selenium fallback")
print("=" * 60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

capgemini_jobs = []

# Try Workday first (Capgemini uses Workday)
try:
    capgemini_jobs = scrape_workday(
        tenant="capgemini",
        instance="wd3",
        career_site="Global",
        company_name="Capgemini",
        industry="IT Services & Consulting",
        location_filter=LOCATION_FILTER,
        max_jobs=500
    )
except Exception as e:
    print(f"  Workday approach failed: {e}")

# Fallback: Selenium on capgemini.com/in-en/careers
if len(capgemini_jobs) < 5:
    print("\n  Trying Selenium on capgemini.com careers...")
    driver = setup_selenium()
    try:
        url = "https://www.capgemini.com/in-en/careers/join-capgemini/job-search/?search_term=&country=in"
        driver.get(url)
        time.sleep(8)

        for page in range(10):
            soup = BeautifulSoup(driver.page_source, "lxml")
            cards = soup.select("article.job-result, .job-card, [class*='job-result'], [class*='job-listing']")
            if not cards:
                cards = soup.select("a[href*='/careers/'], a[href*='/job/']")

            for card in cards:
                title_el = card.select_one("h2, h3, h4, .job-title, [class*='title']")
                title = title_el.get_text(strip=True) if title_el else ""
                loc_el = card.select_one(".location, [class*='location'], [class*='city']")
                loc = loc_el.get_text(strip=True) if loc_el else "India"
                href = card.get("href", "") if card.name == "a" else ""
                if not href:
                    link = card.select_one("a[href]")
                    href = link.get("href", "") if link else ""

                if title and title not in [j["title"] for j in capgemini_jobs]:
                    capgemini_jobs.append({
                        "job_id": href.split("/")[-1] if href else str(len(capgemini_jobs)),
                        "title": title,
                        "company_name": "Capgemini",
                        "raw_jd_text": card.get_text(" ", strip=True),
                        "location_city": loc.split(",")[0].strip(),
                        "industry": "IT Services & Consulting",
                        "date_posted": datetime.now().strftime("%Y-%m-%d"),
                        "is_active": True,
                        "job_url": href if href.startswith("http") else f"https://www.capgemini.com{href}" if href else "",
                        "business_unit": "",
                        "source_platform": "Capgemini Selenium fallback",
                    })

            try:
                next_btn = driver.find_element(By.CSS_SELECTOR, "a.next, [class*='next'], [rel='next']")
                driver.execute_script("arguments[0].click();", next_btn)
                time.sleep(3)
            except:
                break
    except Exception as e:
        print(f"  Selenium failed: {e}")
    finally:
        driver.quit()

print(f"Total Capgemini India jobs: {len(capgemini_jobs)}")


CAPGEMINI INDIA JOB SCRAPER
Primary: Workday API + Selenium fallback
  Scraping Capgemini via Workday API: https://capgemini.wd3.myworkdayjobs.com/wday/cxs/capgemini/Global/jobs
  Mode: BROAD (no location filter — fetching all global jobs)


  [ERROR] HTTP 422 at offset 0
  Total Capgemini India jobs: 0

  Trying Selenium on capgemini.com careers...


Total Capgemini India jobs: 0


In [5]:
df_capgemini = save_results(capgemini_jobs, "Capgemini", OUTPUT_DIR)
if df_capgemini is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_capgemini.columns]
    print(df_capgemini[cols].head(10).to_string())


  [WARN] No jobs found for Capgemini
